# 01 — Vision Transformer (ViT) dari Nol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/05-vision-transformer/01_vit_dari_nol.ipynb)

**Transformer** awalnya dirancang untuk teks (2017). Pada 2020, paper *"An Image is Worth 16×16 Words"* menunjukkan bahwa gambar pun bisa diproses dengan cara yang sama: **potong gambar menjadi patch, perlakukan setiap patch seperti sebuah kata (token)**, lalu proses dengan Transformer.

Isi notebook:
1. **Patch embedding**: dari gambar menjadi deretan token.
2. **CLS token** dan **positional embedding**.
3. **Self-attention** dan **multi-head attention** dari nol, diverifikasi terhadap implementasi PyTorch.
4. **Transformer encoder block**.
5. Melatih **mini-ViT** di FashionMNIST (dataset modul 01) dan membandingkannya dengan MLP/CNN.
6. Visualisasi **attention map** dan **positional embedding** yang dipelajari.

Konsep attention di sini adalah fondasi modul 06 (Transformers NLP) dan modul-modul LLM berikutnya.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Setup dan Data

Kita memakai **FashionMNIST** (28×28, grayscale, 10 kelas), dataset yang sama dengan modul 01, supaya hasilnya bisa dibandingkan langsung dengan MLP di sana.

`QUICK_RUN=1` (environment variable) memakai subset kecil dan 1 epoch untuk uji cepat di CPU.

In [ ]:
import os
import math
import time
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
BATCH_SIZE = 128
EPOCHS = 1 if QUICK_RUN else 15
DATA_DIR = "./data"

MEAN, STD = 0.2860, 0.3530
train_tf = transforms.Compose([
    transforms.RandomCrop(28, padding=2),   # augmentasi ringan: geser sedikit
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])

full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=train_tf)
test_set = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=eval_tf)
CLASS_NAMES = full_train.classes

train_set, val_set = random_split(full_train, [54000, 6000], generator=torch.Generator().manual_seed(42))
if QUICK_RUN:
    train_set, val_set, test_set = Subset(train_set, range(2000)), Subset(val_set, range(500)), Subset(test_set, range(500))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, num_workers=2)
print(f"device: {device} | train {len(train_set)} | val {len(val_set)} | test {len(test_set)} | epochs {EPOCHS}")

## 2. Patch Embedding: Gambar → Token

Transformer menerima **deretan vektor** (seperti deretan kata). Caranya mengubah gambar:

1. Potong gambar `28×28` menjadi patch `7×7` → `(28/7)² = 16` patch.
2. Ratakan (*flatten*) setiap patch menjadi vektor `7×7 = 49` angka.
3. Proyeksikan dengan layer `Linear` ke dimensi `D` (misalnya 64). Hasilnya **16 token berdimensi 64**.

Untuk ViT-B/16 pada gambar 224×224: patch 16×16 → `(224/16)² = 196` token.

In [ ]:
PATCH = 7
img, label = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())[0]
n_side = 28 // PATCH

# unfold: potong menjadi patch tanpa tumpang tindih
patches = img.unfold(1, PATCH, PATCH).unfold(2, PATCH, PATCH)        # (1, 4, 4, 7, 7)
patches = patches.reshape(-1, PATCH * PATCH)                         # (16, 49)
print("gambar:", tuple(img.shape), "→ patch:", tuple(patches.shape), "(16 token × 49 piksel)")

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(1, 3, 1)
ax.imshow(img[0], cmap="gray")
for k in range(1, n_side):
    ax.axhline(k * PATCH - 0.5, color="red"); ax.axvline(k * PATCH - 0.5, color="red")
ax.set_title(f"{CLASS_NAMES[label]}: grid patch {PATCH}×{PATCH}"); ax.axis("off")
for i in range(16):
    a = fig.add_subplot(4, 12, (i // 4) * 12 + 5 + (i % 4) * 2)
    a.imshow(patches[i].reshape(PATCH, PATCH), cmap="gray", vmin=0, vmax=1)
    a.set_title(str(i), fontsize=7); a.axis("off")
plt.show()

### 2.1 Trik implementasi: patch embedding = konvolusi

"Potong jadi patch lalu `Linear`" **identik** dengan `Conv2d` yang `kernel_size = stride = PATCH`: kernel tidak pernah tumpang tindih, dan setiap posisi kernel mengalikan satu patch dengan bobot yang sama. Hampir semua implementasi ViT memakai `Conv2d` karena lebih cepat.

Kita buktikan dengan menyalin bobot `Conv2d` ke `Linear`, lalu membandingkan hasilnya.

In [ ]:
D = 64
conv = nn.Conv2d(1, D, kernel_size=PATCH, stride=PATCH)
linear = nn.Linear(PATCH * PATCH, D)
with torch.no_grad():
    linear.weight.copy_(conv.weight.reshape(D, -1))   # (64, 1, 7, 7) -> (64, 49)
    linear.bias.copy_(conv.bias)

x = img.unsqueeze(0)                                    # (1, 1, 28, 28)
out_conv = conv(x).flatten(2).transpose(1, 2)            # (1, 64, 4, 4) -> (1, 16, 64)
out_linear = linear(patches.unsqueeze(0))                # (1, 16, 64)
print("conv  :", tuple(out_conv.shape))
print("linear:", tuple(out_linear.shape))
print("identik?", torch.allclose(out_conv, out_linear, atol=1e-6))

## 3. CLS Token dan Positional Embedding

Dua tambahan sebelum masuk Transformer:

- **CLS token**: satu vektor yang **dipelajari**, diletakkan di depan deretan patch (jadi 1 + 16 = 17 token). Lewat attention, token ini "mengumpulkan" informasi dari seluruh patch, dan output-nya di akhir dipakai untuk klasifikasi. Idenya diambil dari BERT (modul 06).
- **Positional embedding**: attention sendiri **tidak tahu urutan/posisi**; mengacak patch tidak mengubah hasilnya. Karena itu setiap posisi diberi vektor yang dipelajari dan **ditambahkan** ke token-nya, sehingga model tahu patch mana di kiri atas dan mana di kanan bawah.

In [ ]:
B, N = 2, 16
tokens = torch.randn(B, N, D)                           # output patch embedding
cls_token = nn.Parameter(torch.zeros(1, 1, D))           # dipelajari
pos_embed = nn.Parameter(torch.randn(1, 1 + N, D) * 0.02)  # dipelajari, satu per posisi

x = torch.cat([cls_token.expand(B, -1, -1), tokens], dim=1)   # (B, 17, D)
x = x + pos_embed                                             # broadcasting di dimensi batch
print("setelah CLS + posisi:", tuple(x.shape))

## 4. Self-Attention

Setiap token membuat tiga vektor lewat proyeksi linear:
- **Query (Q)**: "informasi apa yang saya cari?"
- **Key (K)**: "informasi apa yang saya punya?"
- **Value (V)**: "isi yang akan saya bagikan."

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

1. $QK^\top$: skor kecocokan setiap pasangan token (matriks `N×N`).
2. Dibagi $\sqrt{d_k}$ agar skor tidak terlalu besar (softmax jadi terlalu "tajam" dan gradien mengecil).
3. **Softmax per baris** → bobot attention, dan setiap baris berjumlah 1.
4. Output setiap token = rata-rata tertimbang dari semua **V**.

Artinya, setiap token bisa langsung "melihat" **semua** token lain dalam satu langkah. Bandingkan dengan CNN yang hanya melihat tetangga dekat (kernel 3×3).

### 4.1 Contoh angka kecil

In [ ]:
torch.manual_seed(0)
n_tok, d_k = 3, 4
Q, K, V = torch.randn(n_tok, d_k), torch.randn(n_tok, d_k), torch.randn(n_tok, d_k)

skor = Q @ K.T / math.sqrt(d_k)          # (3, 3)
bobot = torch.softmax(skor, dim=-1)      # tiap baris jumlahnya 1
out = bobot @ V                          # (3, 4)

torch.set_printoptions(precision=3, sci_mode=False)
print("skor QKᵀ/√d:\n", skor)
print("bobot attention (softmax per baris):\n", bobot)
print("jumlah per baris:", bobot.sum(-1))
print("output:\n", out)

ref = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
print("\nsama dengan F.scaled_dot_product_attention?", torch.allclose(out, ref, atol=1e-6))

### 4.2 Multi-Head Attention

Satu attention hanya bisa fokus pada satu "jenis" hubungan. **Multi-head attention** menjalankan beberapa attention secara paralel (misalnya 4 head), masing-masing di sub-ruang berdimensi `D / n_heads`. Satu head bisa fokus ke bentuk, head lain ke tekstur, dan seterusnya. Hasil semua head digabung lalu diproyeksikan kembali ke dimensi `D`.

Implementasi di bawah menyimpan bobot attention terakhir di `self.last_attn` untuk divisualisasikan nanti. Kita verifikasi hasilnya terhadap `nn.MultiheadAttention` bawaan PyTorch dengan menyalin bobot yang sama.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads, self.d_head = n_heads, dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim)      # Q, K, V dihitung sekaligus
        self.proj = nn.Linear(dim, dim)         # gabungan semua head -> dim
        self.last_attn = None

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                        # masing-masing (B, heads, N, d_head)
        attn = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d_head), dim=-1)  # (B, heads, N, N)
        self.last_attn = attn.detach()
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)      # gabungkan head
        return self.proj(out)


mha = MultiHeadAttention(D, n_heads=4)
ref = nn.MultiheadAttention(D, num_heads=4, batch_first=True)
with torch.no_grad():   # salin bobot agar keduanya identik
    ref.in_proj_weight.copy_(mha.qkv.weight); ref.in_proj_bias.copy_(mha.qkv.bias)
    ref.out_proj.weight.copy_(mha.proj.weight); ref.out_proj.bias.copy_(mha.proj.bias)

x = torch.randn(2, 17, D)
ours = mha(x)
theirs, _ = ref(x, x, x)
print("output:", tuple(ours.shape), "| bobot attention:", tuple(mha.last_attn.shape))
print("sama dengan nn.MultiheadAttention?", torch.allclose(ours, theirs, atol=1e-5))

## 5. Transformer Encoder Block

Satu blok encoder (varian *pre-norm* yang dipakai ViT):

```
x = x + MultiHeadAttention(LayerNorm(x))   # token saling bertukar informasi
x = x + MLP(LayerNorm(x))                  # tiap token diproses sendiri-sendiri
```

- **Residual connection** (`x + ...`): seperti ResNet, memudahkan training jaringan dalam.
- **LayerNorm**: menormalkan setiap token (bukan per batch seperti BatchNorm).
- **MLP**: `Linear → GELU → Linear`, biasanya melebar 2–4× di tengah.

ViT = patch embedding + CLS + posisi → **L blok encoder** → ambil output CLS → `Linear` ke jumlah kelas.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, dim, n_heads, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class MiniViT(nn.Module):
    def __init__(self, img_size=28, patch=7, in_ch=1, n_classes=10, dim=64, depth=4, n_heads=4):
        super().__init__()
        self.n_patches = (img_size // patch) ** 2
        self.patch_embed = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + self.n_patches, dim) * 0.02)
        self.blocks = nn.ModuleList([EncoderBlock(dim, n_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)              # (B, N, dim)
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], dim=1)
        x = x + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x[:, 0]))                           # hanya token CLS


model = MiniViT().to(device)
print(f"Jumlah parameter MiniViT: {sum(p.numel() for p in model.parameters()):,}")

### 5.1 Melacak ukuran tensor

Perhatikan: setelah patch embedding, **ukuran tensor tidak berubah** di sepanjang blok encoder (`B × 17 × 64`). Berbeda dengan CNN yang resolusinya mengecil di setiap tahap.

In [ ]:
x = torch.randn(1, 1, 28, 28, device=device)
with torch.no_grad():
    t = model.patch_embed(x);                          print(f"{'patch_embed (Conv2d)':<24} {tuple(t.shape)}")
    t = t.flatten(2).transpose(1, 2);                  print(f"{'flatten + transpose':<24} {tuple(t.shape)}")
    t = torch.cat([model.cls_token, t], dim=1);        print(f"{'+ CLS token':<24} {tuple(t.shape)}")
    t = t + model.pos_embed;                           print(f"{'+ positional embedding':<24} {tuple(t.shape)}")
    for i, blk in enumerate(model.blocks):
        t = blk(t);                                    print(f"{f'encoder block {i}':<24} {tuple(t.shape)}")
    t = model.head(model.norm(t[:, 0]));               print(f"{'CLS → head':<24} {tuple(t.shape)}")

## 6. Training

Resep training yang umum untuk Transformer:
- **AdamW** dengan *weight decay*.
- **Warmup + cosine**: learning rate naik perlahan di awal (Transformer sensitif pada langkah-langkah awal), lalu turun mengikuti kurva cosinus. Scheduler dipanggil **setiap langkah** (batch).
- **Label smoothing** 0.1: target tidak 100% satu kelas, sehingga model tidak terlalu yakin (regularisasi).

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    model.train()
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if scheduler is not None:
            scheduler.step()  # scheduler per langkah (bukan per epoch)
        total_loss += loss.item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data


loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(0.1 * total_steps))

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps                         # warmup linear
    progres = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progres))               # cosine turun ke 0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
mulai = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn)
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} | lr {scheduler.get_last_lr()[0]:.1e}")
print(f"\nWaktu training: {time.time() - mulai:.0f} detik")

## 7. Hasil dan Perbandingan

Dengan 15 epoch penuh, mini-ViT ini biasanya mencapai akurasi test sekitar **88–90%**, kira-kira setara dengan MLP di modul 01 dan sedikit di bawah CNN kecil (umumnya 91–93%). Mengapa ViT tidak langsung menang?

- CNN punya **inductive bias** bawaan: lokalitas (kernel kecil) dan *translation equivariance*. ViT harus **mempelajari** hubungan spasial sendiri dari data.
- Karena itu ViT butuh **data jauh lebih banyak** (atau pretraining di dataset besar) untuk mengungguli CNN. Keunggulan ViT baru terlihat pada skala besar, dan itulah sebabnya di notebook 02 kita memakai ViT **pretrained**.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Akurasi test MiniViT: {test_acc:.3f}")

ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, history["train_loss"], "o-", label="train"); ax[0].plot(ep, history["val_loss"], "o-", label="val")
ax[1].plot(ep, history["train_acc"], "o-", label="train"); ax[1].plot(ep, history["val_acc"], "o-", label="val")
for a, t in zip(ax, ["Loss", "Akurasi"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 8. Visualisasi Attention

Bobot attention dari **token CLS ke setiap patch** di blok terakhir menunjukkan patch mana yang paling "diperhatikan" model saat memutuskan kelas. Kita rata-ratakan semua head, susun ulang menjadi grid `4×4`, lalu tampilkan di atas gambar.

Model yang sudah terlatih cenderung memperhatikan patch yang berisi objek (bukan background hitam).

In [ ]:
model.eval()
raw = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=eval_tf)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
with torch.no_grad():
    for j in range(6):
        x, y = raw[j]
        pred = model(x.unsqueeze(0).to(device)).argmax(1).item()
        attn = model.blocks[-1].attn.last_attn[0]           # (heads, 17, 17)
        cls_ke_patch = attn[:, 0, 1:].mean(0).reshape(n_side, n_side).cpu()
        axes[0, j].imshow(x[0] * STD + MEAN, cmap="gray")
        axes[0, j].set_title(f"{CLASS_NAMES[y]}\npred: {CLASS_NAMES[pred]}", fontsize=8)
        axes[1, j].imshow(x[0] * STD + MEAN, cmap="gray")
        axes[1, j].imshow(cls_ke_patch, cmap="jet", alpha=0.5, extent=(-0.5, 27.5, 27.5, -0.5))
        axes[1, j].set_title("attention CLS → patch", fontsize=8)
for a in axes.flat:
    a.axis("off")
plt.tight_layout(); plt.show()

## 9. Positional Embedding yang Dipelajari

Setiap posisi punya vektor sendiri. Jika kita hitung **cosine similarity** antara positional embedding satu patch dengan semua patch lain, model yang terlatih biasanya menunjukkan pola spasial: patch yang **berdekatan** (baris/kolom sama) punya embedding yang lebih mirip. Model menemukan struktur 2D gambar sendiri, padahal kita hanya memberinya daftar 16 posisi tanpa informasi baris/kolom.

(Dengan `QUICK_RUN` yang hanya 1 epoch, pola ini belum terbentuk.)

In [ ]:
pos = model.pos_embed[0, 1:].detach().cpu()        # (16, dim), tanpa CLS
sim = F.cosine_similarity(pos[:, None, :], pos[None, :, :], dim=-1)   # (16, 16)

fig, axes = plt.subplots(n_side, n_side, figsize=(7, 7))
for i, a in enumerate(axes.flat):
    a.imshow(sim[i].reshape(n_side, n_side), cmap="viridis", vmin=-1, vmax=1)
    a.set_title(f"patch {i}", fontsize=7); a.set_xticks([]); a.set_yticks([])
fig.suptitle("Cosine similarity positional embedding: patch i vs semua patch")
plt.tight_layout(); plt.show()

## Ringkasan

- ViT memotong gambar menjadi **patch**, memproyeksikannya menjadi **token** (setara `Conv2d` dengan `kernel = stride = patch`), menambah **CLS token** dan **positional embedding**, lalu memprosesnya dengan blok **Transformer encoder**.
- **Self-attention**: `softmax(QKᵀ/√d)·V`; setiap token bisa melihat semua token lain dalam satu langkah.
- **Multi-head** menjalankan beberapa attention paralel di sub-ruang berbeda.
- Tanpa pretraining dan pada data kecil, ViT belum tentu mengalahkan CNN karena tidak punya inductive bias spasial; kekuatannya muncul pada skala besar.

## Latihan

1. **Ukuran patch.** Latih MiniViT dengan patch 4 (49 token), 7 (16 token), dan 14 (4 token) dengan jumlah epoch yang sama. Catat akurasi, jumlah parameter, dan waktu per epoch. Jelaskan trade-off-nya (petunjuk: biaya attention sebanding dengan N²).
2. **Tanpa informasi posisi.** Latih ulang MiniViT tanpa `pos_embed` (ganti dengan nol yang tidak dilatih). Lalu, pada model **dengan** positional embedding, acak urutan patch gambar test sebelum masuk model (permutasi yang sama untuk semua gambar) dan ukur akurasinya. Jelaskan hasil kedua eksperimen.
3. **ViT vs CNN saat data sedikit.** Latih MiniViT dan CNN kecil (misalnya 2 blok konvolusi seperti modul 02) memakai hanya 10% dan 100% data train. Buat tabel akurasi test untuk keempat kombinasi. Apakah hasilnya mendukung penjelasan tentang inductive bias di bagian 7?